# Mushroom Dataset - Cleaning

## Contributors
- Pamela: cleaning and preparation

Now that we’ve looked the raw dataset, we will start cleaning it up for modelling.

In this notebook we’ll:
- remove columns that don’t look useful
- decide what to do with missing values
- keep or remove outliers where needed
- prepare the categorical columns
- make sure the final dataset still looks normal after cleaning

In [1]:
import pandas as pd

df = pd.read_csv("mushroom_project_dataset.csv")

df.head()

,cap-diameter,stem-height,stem-width,spore-print-color,gill-color,habitat,season,ring-type,cap-shape,stem-surface,jumbled_noise_0,jumbled_noise_1,class
0,NaN,6.42,13.51,NaN,NaN,d,u,g,x,NaN,x,c,p
1,6.99,7.68,18.76,NaN,w,d,a,f,f,s,f,f,e
2,5.78,7.67,8.21,NaN,r,d,u,f,c,NaN,c,x,p
3,6.34,8.34,11.06,NaN,NaN,d,a,z,x,NaN,x,b,p
4,NaN,5.67,12.78,NaN,n,d,NaN,f,x,NaN,s,s,e


In [2]:
df = df.drop(
    columns=[
        "jumbled_noise_0",
        "jumbled_noise_1"
    ]
)

### Removing the noise columns

The two `jumbled_noise` columns didn’t show a clear useful pattern in the EDA.

Their edible/poisonous split stayed fairly close to the overall class split, so they don’t seem very useful for prediction.

Because of that, we’ll remove them before modelling.

In [3]:
df = df.drop(columns=["spore-print-color"])

### Removing spore-print-color

Around 95% of this column is missing.

Even though the available values showed some useful patterns in the EDA, there are too few values available for most mushrooms.

For now, we'll remove this column. We can later test if keeping it improves the model.

In [4]:
df.isna().mean().sort_values(ascending=False) * 100

stem-surface    67.18
cap-diameter    40.64
stem-height     20.26
habitat         20.08
gill-color      19.78
season          19.46
ring-type       11.54
cap-shape        8.12
stem-width       7.90
class            0.00
dtype: float64

In [5]:
categorical_columns = [
    "gill-color",
    "habitat",
    "season",
    "ring-type",
    "cap-shape",
    "stem-surface"
]

for column in categorical_columns:
    df[column] = df[column].fillna("unknown")

### Missing categorical values

For the categorical columns, we’ll replace missing values with `unknown`.

This way we don’t have to delete a lot of rows, and we also don’t pretend that the missing value is actually one of the existing categories.

In [6]:
numeric_columns = [
    "cap-diameter",
    "stem-height",
    "stem-width"
]

### Missing numerical values

The numerical columns still contain missing values.

We won't fill these values yet, because the median should only be calculated using the training data. Doing it before splitting the dataset could cause data leakage.

We will handle these missing values later in the modelling pipeline.

### Outliers

There were quite a few outliers in the numerical features, but they didn’t look like obvious mistakes.

Some of the extreme values still looked like valid mushrooms, so we’ll keep them for now instead of deleting them just because they fall outside the boxplot.

In [7]:
df.shape

(5000, 10)

In [8]:
df.isna().sum()

cap-diameter    2032
stem-height     1013
stem-width       395
gill-color         0
habitat            0
season             0
ring-type          0
cap-shape          0
stem-surface       0
class              0
dtype: int64

In [9]:
df.head()

,cap-diameter,stem-height,stem-width,gill-color,habitat,season,ring-type,cap-shape,stem-surface,class
0,NaN,6.42,13.51,unknown,d,u,g,x,unknown,p
1,6.99,7.68,18.76,w,d,a,f,f,s,e
2,5.78,7.67,8.21,r,d,u,f,c,unknown,p
3,6.34,8.34,11.06,unknown,d,a,z,x,unknown,p
4,NaN,5.67,12.78,n,d,unknown,f,x,unknown,e


In [10]:
df["class"].value_counts(normalize=True)

class
e    0.6214
p    0.3786
Name: proportion, dtype: float64

### Quick check after cleaning

The columns we decided not to use are now removed and the categorical missing values are handled.

We kept all 5,000 rows, so no mushrooms were removed during cleaning.

The numerical columns still contain some missing values. We leave these for the modelling step, where they can be filled using only the training data.

The edible/poisonous split also stayed the same.

In [11]:
df.to_csv("mushroom_cleaned.csv", index=False)